In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler

In [3]:
base_path = Path("..")

processed_path = base_path / "data" / "processed"

df = pd.read_csv(processed_path / "model_data_monthly.csv")

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(["date","ticker"]).reset_index(drop=True)

df.head()

,date,month,ticker,mom_1m,mom_3m,mom_6m,vol_1m,vol_3m,vol_6m,volume_ratio,fwd_ret_1m
0,2018-07-31,2018-07,AKAM,0.027721,0.062394,0.130709,0.015373,0.015739,0.017518,1.124732,-0.000399
1,2018-07-31,2018-07,CHKP,0.153460,0.144090,0.079111,0.017856,0.012881,0.013422,2.060624,0.044466
2,2018-07-31,2018-07,CSCO,-0.009537,-0.049306,0.016858,0.013935,0.012183,0.016595,0.941895,0.122724
3,2018-07-31,2018-07,CYBR,-0.035737,0.103417,0.424449,0.022415,0.017598,0.017860,1.864596,0.214956
4,2018-07-31,2018-07,FTNT,0.007689,0.128633,0.375383,0.018415,0.016216,0.017128,1.275973,0.311556


In [4]:
feature_cols = [
    "mom_1m",
    "mom_3m",
    "mom_6m",
    "vol_1m",
    "vol_3m",
    "vol_6m",
    "volume_ratio"
]

target_col = "fwd_ret_1m"

In [5]:
dates = sorted(df["date"].unique())

len(dates)

89

In [6]:
results = []

train_window = 24

In [7]:
for i in range(train_window, len(dates)-1):

    train_dates = dates[i-train_window:i]
    test_date = dates[i]

    train = df[df["date"].isin(train_dates)].dropna()
    test = df[df["date"] == test_date].dropna()

    if len(test) == 0 or len(train) == 0:
        continue

    X_train = train[feature_cols]
    y_train = train[target_col]

    X_test = test[feature_cols]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42)

    model.fit(X_train_scaled, y_train)

    test = test.copy()

    test["pred_ret"] = model.predict(X_test_scaled)

    results.append(test)

In [8]:
predictions = pd.concat(results).reset_index(drop=True)

predictions.head()

,date,month,ticker,mom_1m,mom_3m,mom_6m,vol_1m,vol_3m,vol_6m,volume_ratio,fwd_ret_1m,pred_ret
0,2020-07-31,2020-07,AKAM,0.057463,0.172961,0.204499,0.021498,0.019408,0.029423,0.620392,0.035486,0.013024
1,2020-07-31,2020-07,CHKP,0.161509,0.229042,0.096579,0.016451,0.015150,0.027215,1.186718,0.007260,-0.000977
2,2020-07-31,2020-07,CRWD,0.099563,0.635602,0.853004,0.037055,0.034493,0.045386,0.434706,0.110689,0.050410
3,2020-07-31,2020-07,CSCO,0.030634,0.160094,0.042443,0.013144,0.019398,0.033450,0.779323,-0.103609,0.015954
4,2020-07-31,2020-07,CYBR,0.145301,0.222914,-0.147508,0.024731,0.029831,0.040405,0.711850,-0.062288,0.045840


In [9]:
predictions.to_csv(processed_path / "monthly_predictions.csv", index=False)

print("Saved predictions.")

Saved predictions.
